# 📘 Pydantic for AI Engineers

## 1. What is Pydantic?

Pydantic is a parsing and validation framework for defining specialized Python classes. It leverages standard Python type hints to ensure data conforms to specific structures and types.

* **Model:** A specialized Pydantic class that inherits from `BaseModel`.
* **Fields:** The class attributes defined within a model.
* **The Golden Rule:** Pydantic guarantees that the types and constraints of the resulting model instance match your definitions. It does the heavy lifting of data validation so you do not have to write boilerplate `if not isinstance(val, str):` checks.

## 2. Pydantic vs. Dataclasses

A common point of confusion is whether to use Dataclasses or Pydantic. They look similar syntactically but serve entirely different core purposes.

| Feature | Standard `dataclass` | Pydantic `BaseModel` |
| --- | --- | --- |
| **Core Mechanism** | Code generator (writes `__init__`, `__repr__` for you). | Inheritance framework (inherits from `BaseModel`). |
| **Validation** | ❌ None out-of-the-box. | ✅ Robust, automatic type checking and validation. |
| **Serialization** | ❌ Manual (requires custom dict/JSON methods). | ✅ Built-in (`model_dump()`, `model_dump_json()`). |
| **Deserialization** | ❌ Manual. | ✅ Built-in (`model_validate()`, `model_validate_json()`). |
| **Performance** | Faster (standard Python objects). | Slightly slower, though V2 is powered by Rust core and highly optimized. |

**When to use which?** * Use **Dataclasses** for internal data structures where performance is critical, and data is already trusted/clean.

* Use **Pydantic** whenever data is crossing a boundary (e.g., API requests in FastAPI, parsing LLM outputs in LangChain, reading JSON from a database).

## 3. The Core Lifecycle

Pydantic primarily exists to handle the flow of data in and out of your applications.

1. **Deserialization & Validation (Input):** Taking raw data (like a Dictionary or JSON string) and converting it into a Python object. During this step, Pydantic actively validates types and constraints.
2. **Manipulation:** Working with the data using standard Python object dot-notation (e.g., `user.first_name`).
3. **Serialization (Output):** Converting the Python object back into a Dictionary or JSON string to send over a network (e.g., returning an API response).

---

## 4. Basic Implementation

### Defining and Instantiating a Model

```python
from pydantic import BaseModel, ValidationError

# 1. Defining the Model
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

# 2. Creating an instance using keyword arguments
p = Person(first_name="Evariste", last_name="Galois", age=20)

# Pydantic automatically generates a readable __str__ and __repr__
print(p)
# Output: first_name='Evariste' last_name='Galois' age=20

# 3. Accessing and Modifying Fields
print(p.first_name) # Evariste
p.age = 21          # Models are mutable by default
```

In [1]:
from pydantic import BaseModel

In [3]:
# Defining the Model
class Person(BaseModel):
    first_name : str
    last_name : str
    age : int

In [4]:
p = Person(first_name="Vijaya Simha", last_name="M", age=20)

In [9]:
try :
    q= Person(first_name = "user", last_name = "s", age = "twenty")
except Exception as e:
    print(e)

1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [11]:
try :
    q= Person(first_name = "user", last_name = "s", age = "20")
except Exception as e:
    print(e)

In [12]:
print(q)

first_name='user' last_name='s' age=20


### Handling Validation Errors

Pydantic collects **all** validation errors before raising an exception, rather than failing at the first bad field. This is crucial for returning complete error payloads in APIs.

```python
try:
    # Missing first_name and age, plus passing a list instead of a string
    bad_person = Person(last_name=["Galois"])
except ValidationError as e:
    print(e)
    """
    Output will list ALL errors:
    - first_name: Field required
    - last_name: Input should be a valid string
    - age: Field required
    """

```

# Deserialization & Serialization

## 1. Deserialization (Input / Loading Data)

Deserialization is the act of taking raw data (like a dictionary or JSON string) and converting it into a Pydantic model instance. During this process, Pydantic parses the data, coerces types if necessary, and validates constraints.

### Basic Deserialization Methods

While you *can* use dictionary unpacking (`Person(my_dict)`), it is highly discouraged for complex or nested models. Instead, use Pydantic's built-in `model_` methods.

* **`model_validate(obj)`:** Takes a Python dictionary (or another object) and loads it into the model.
* **`model_validate_json(json_string)`:** Takes a raw JSON string. This is heavily used in FastAPI and API integrations because it skips the intermediate step of manually running `json.loads()`.

```python
from pydantic import BaseModel, ValidationError

class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

# 1. From a Dictionary
raw_dict = {"first_name": "Alan", "last_name": "Turing", "age": 41}
p1 = Person.model_validate(raw_dict)

# 2. From a JSON String
raw_json = '{"first_name": "Ada", "last_name": "Lovelace", "age": 36}'
p2 = Person.model_validate_json(raw_json)

```

> **Note on JSON Validation:** JSON is strictly typed regarding formatting. A trailing comma in your JSON string (e.g., `{"age": 30,}`) will cause `model_validate_json` to raise a `ValidationError` because it is mathematically invalid JSON.

### 🚀 Advanced Deserialization Concepts

**A. Strict Mode**
By default, Pydantic tries to coerce types (e.g., if you pass the string `"42"` to an `int` field, it converts it to `42`). In financial or strict API environments, you want to fail if the type isn't exact.

```python
# Fails if passed {"age": "41"} instead of {"age": 41}
p_strict = Person.model_validate(raw_dict, strict=True)

```

**B. AliasChoices (Crucial for LLM Outputs)**
LLMs often hallucinate key names. Sometimes they return `first_name`, sometimes `firstName`, sometimes `GivenName`. You can use `AliasChoices` to accept any of them during deserialization.

```python
from pydantic import Field, AliasChoices

class FlexiblePerson(BaseModel):
    first_name: str = Field(validation_alias=AliasChoices('first_name', 'firstName', 'GivenName'))

```



In [2]:
from pydantic import BaseModel, ValidationError

In [2]:
class Person(BaseModel):
    first_name: str
    last_name: str
    salary : int


In [5]:
vijay = Person("vijay", "M", 50000)

TypeError: BaseModel.__init__() takes 1 positional argument but 4 were given

### Explanation : 
Pydantic's BaseModel does not accept positional arguments by default; all fields must be passed as keyword arguments. The error occurs because you are passing 3 positional values, but Pydantic's __init__ only expects self

In [3]:
vijay = Person(first_name ="vijay", last_name ="M", salary ="100000")
print(vijay)

first_name='vijay' last_name='M' salary=100000


In [9]:
# Dictionary unpacking (same as above object creation)
ajay_details = {"first_name" : "Ajay",
               "last_name" : "D",
               "salary" : 20000}
ajay = Person(**ajay_details)
print(ajay)

first_name='Ajay' last_name='D' salary=20000


In [11]:
ajay_1 = Person.model_validate(ajay_details)
print(ajay_1)

first_name='Ajay' last_name='D' salary=20000


In [16]:
ajay_json = '''{
"first_name" : "Ajay",
"last_name" : "D",
"salary" : 20000}
'''
ajay_2 = Person.model_validate_json(ajay_json)
print(ajay_2)

first_name='Ajay' last_name='D' salary=20000


In [18]:
invalid_json = ajay_json = '''{
"first_name" : "Ajay",
"last_name" : "D",}
'''
try :
    Person.model_validate_json(invalid_json)
except ValidationError as e :
    print(e)
# pydantic uses strict json parser and the JSON specification does not allow trailing commas.

1 validation error for Person
  Invalid JSON: trailing comma at line 3 column 19 [type=json_invalid, input_value='{\n"first_name" : "Ajay",\n"last_name" : "D",}\n', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid


In [19]:
missing_json = '''{
"first_name" : "Ajay"
} '''

try : 
    Person.model_validate_json(missing_json)
except ValidationError as e :
    print(e)


2 validation errors for Person
last_name
  Field required [type=missing, input_value={'first_name': 'Ajay'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
salary
  Field required [type=missing, input_value={'first_name': 'Ajay'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


# How Pydantic Returns All Validation Errors at Once

Pydantic doesn't stop at the first error. Instead, it **parses the entire input**, checks every field against the model, **collects all validation errors**, and raises **one `ValidationError`** containing every issue it found.

In your example:

```python
missing_json = '''{
    "first_name": "Ajay"
}'''

Person.model_validate_json(missing_json)
```

If the model is:

```python
class Person(BaseModel):
    first_name: str
    last_name: str
    salary : int
```

Pydantic notices that `last_name and salary` is missing. Rather than raising an exception immediately, it continues validating the remaining fields, gathers any additional errors, and finally raises a single `ValidationError` with the complete list of problems.

Internally, the Rust-based **pydantic-core** validation engine accumulates errors during validation instead of failing fast. This lets developers see all validation issues in one run, making debugging and user feedback much more efficient.


## 2. Serialization (Output / Exporting Data)

Serialization is the reverse: taking a valid Python Pydantic model and dumping it back out into a raw dictionary or JSON string to send over the wire.

### Basic Serialization Methods

* **`model_dump()`:** Converts the model into a standard Python dictionary.
* **`model_dump_json()`:** Converts the model into a JSON string. You can pass standard `json` module kwargs here (like `indent=2` for pretty printing).

```python
# Convert to dictionary
person_dict = p1.model_dump()

# Convert to JSON String
person_json = p1.model_dump_json(indent=2)
print(person_json)

```

### 🚀 Advanced Serialization Concepts

**A. Inclusion and Exclusion**
You rarely want to send your entire database model to the frontend (e.g., you want to hide passwords or internal IDs).

```python
# Only export specific fields
p1.model_dump(include={'first_name', 'last_name'})

# Exclude specific fields
p1.model_dump(exclude={'age'})

```

**B. `exclude_unset` (Crucial for HTTP PATCH Requests)**
If you have a model with default values, `model_dump()` exports everything. If you are building an API endpoint that updates a database, you only want to send the fields the user *actually provided*.

```python
class UpdateUser(BaseModel):
    name: str = "Unknown"
    age: int = 0

user = UpdateUser(age=25) 
# The user only explicitly set 'age'. 'name' is the default.

print(user.model_dump(exclude_unset=True))
# Output: {'age': 25} -> Perfect for building dynamic SQL UPDATE queries!

```

**C. Exporting by Alias (`by_alias=True`)**
Python uses `snake_case`, but frontend JavaScript frameworks (and many APIs) expect `camelCase`.

```python
class APIUser(BaseModel):
    first_name: str = Field(alias="firstName")

user = APIUser(firstName="Grace")
# Using by_alias=True ensures the output is camelCase, hiding your Pythonic snake_case
print(user.model_dump(by_alias=True)) 
# Output: {'firstName': 'Grace'}

```

In [5]:
Person_dict = vijay.model_dump()
print(Person_dict)
print(type(Person_dict))

{'first_name': 'vijay', 'last_name': 'M', 'salary': 100000}
<class 'dict'>


In [6]:
Person_json = vijay.model_dump_json()
print(Person_json)
print(type(Person_json))

{"first_name":"vijay","last_name":"M","salary":100000}
<class 'str'>


### Deserialization & Serialization: Interview Preparation

<details>
<summary><b>Question 1: I see developers often load dictionary data into a Pydantic model using dictionary unpacking, like `User(my_dict)`. Why is it generally better to use `User.model_validate(my_dict)` instead? <b> (Click to reveal)</summary>
<br>
    
**Answer:** Dictionary unpacking (``) works fine for flat, simple models, but it breaks down completely when you introduce nested models.

If my `User` model contains a list of `Address` Pydantic models, doing `User(my_dict)` will successfully create the User, but the items inside the addresses list will remain raw Python dictionaries. `model_validate()` recursively parses the data, ensuring that all nested dictionaries are properly instantiated into their respective Pydantic sub-models. Plus, `model_validate` is heavily optimized in Pydantic V2's Rust core.
</details>

<details>
<summary><b>Question 2: You are building an HTTP PATCH endpoint where a client can update their profile. They might send just their `age`, or just their `last_name`. How do you serialize the Pydantic model to update the database without accidentally overwriting the missing fields with default values?  <b> (Click to reveal)</summary>
<br>
    
**Answer:** This is a classic use case for the `exclude_unset=True` argument inside `model_dump()`.

When Pydantic deserializes the incoming payload, it keeps track of which fields were explicitly provided by the user and which ones were just filled in with default values. When I call `model_dump(exclude_unset=True)`, it exports a dictionary containing *only* the fields the user actually sent. I can then safely pass that dictionary to my ORM or database layer to perform a partial update.

</details>

<details>
<summary><b>Question 3: You are parsing JSON output from an LLM. The LLM is mostly consistent, but sometimes it returns the key `first_name`, and other times it returns `firstName`. How do you handle this cleanly in Pydantic?<b> (Click to reveal)</summary>
<br>
    
**Answer:** Instead of writing fragile regex checks or trying to clean the string before it hits Pydantic, I would handle this directly in the model field using `AliasChoices`.

I would define the field as: `first_name: str = Field(validation_alias=AliasChoices('first_name', 'firstName'))`.

During deserialization, Pydantic will check the incoming payload for either of those keys and map it correctly to the `first_name` attribute. This keeps my internal codebase strictly snake_case while easily handling the LLM's slight hallucinations.
</details>

<details>
<summary><b>Question 4: By default, Pydantic tries to be helpful by coercing types. If a user sends `{"age": "25"}` as a string, Pydantic will convert it to the integer 25. How do you stop this and enforce exact type matching?<b> (Click to reveal)</summary>
<br>
    
**Answer:** If I'm building a strict API where I want to reject loose typing, I use Pydantic's strict mode.

I can either enforce this globally on the model by setting `model_config = ConfigDict(strict=True)`, or I can enforce it at the deserialization step by passing the argument directly: `User.model_validate(data, strict=True)`. If strict mode is enabled, passing the string `"25"` to an `int` field will immediately throw a `ValidationError`.
</details>

<details>
<summary><b>Question 5: A junior developer on your team wants to export a Pydantic model to a dictionary, so they just call `my_model.__dict__`. Why is this a bad idea in production, and what should they use instead?<b> (Click to reveal)</summary>
<br>
    
**Answer:** Using `__dict__` is a trap because it only returns the top-level instance attributes. It doesn't perform a recursive export.

If your model has nested Pydantic models, complex objects like `datetime` or `UUID`, or custom `@field_serializer` decorators, `__dict__` ignores all of that. The resulting dictionary might not even be JSON-serializable.

They should always use `model_dump()`. It traverses the entire object tree recursively, applies custom serializers, formats complex data types, and respects inclusion/exclusion and alias rules.
</details>



<details>
<summary><b>Question 6: Python relies on `snake_case`, but most frontend frameworks expect API responses in `camelCase`. How do you handle this mismatch when serializing data to send back to the client?<b> (Click to reveal)</summary>
<br>

 **Answer:** I define my Pydantic fields using standard Python snake_case, but I assign a camelCase alias to them. For example: `last_name: str = Field(alias="lastName")`.

Within my Python logic, I interact with the object normally (`user.last_name`). But when it is time to serialize the response for the frontend, I call `user.model_dump(by_alias=True)`. This tells Pydantic to swap the keys to their aliases, ensuring the frontend receives clean camelCase JSON without polluting my Python codebase.
</details>


# Type Coercion & Strict Mode

## 1. What is Type Coercion?

When deserializing data, the input data types rarely match your model's field types perfectly (especially when parsing JSON, which has limited native types). **Type Coercion** is Pydantic's attempt to transform or cast the incoming data into the expected field type.

```python
from pydantic import BaseModel

class Coordinates(BaseModel):
    x: float
    y: float

# Deserializing with mismatched types
# 'x' is an int, 'y' is a string
p2 = Coordinates.model_validate({"x": 0, "y": "1.2"})

print(type(p2.x)) # <class 'float'> -> Coerced from 0
print(type(p2.y)) # <class 'float'> -> Coerced from "1.2"

```

In **Lax Mode** (the default), Pydantic successfully coerces the integer `0` and the string `"1.2"` into valid floats.

## 2. Lax Mode vs. Strict Mode

Pydantic operates in Lax Mode by default, but you can configure it to be strict.

* **Lax Mode:** Attempts a wide variety of logical type conversions (e.g., string `"yes"` to boolean `True`, or string `"1.2"` to float `1.2`).
* **Strict Mode:** Far less forgiving. If you ask for a `float` and provide an `int`, it will raise a `ValidationError`.

*Note: You can enforce strict mode globally on a model via `ConfigDict(strict=True)` or per validation call via `model_validate(data, strict=True)`.*

## 3. The "String Coercion" Exception (Preventing Silent Bugs)

You might assume that because an integer like `100` can easily be cast to a string `"100"`, Pydantic would allow this in Lax mode. **It does not.**

```python
class TextModel(BaseModel):
    field: str

# This will RAISE a ValidationError!
TextModel(field=100) 

```

**Why doesn't Pydantic coerce ints or dicts to strings?**
In Python, *every* object has a string representation (via `__str__` or `__repr__`). If Pydantic auto-coerced everything to a string, it would hide severe API bugs.

**The Real-World API Scenario:**
Imagine you integrate with a 3rd party API that sends this JSON:
`{"email": "user@example.com"}` (A String).

Months later, the API updates its structure without warning you:
`{"email": {"personal": "user@...", "work": "work@..."}}` (A Dictionary).

If Pydantic allowed dict-to-string coercion, it would quietly load `"{'personal': 'user@...', 'work': 'work@...'}"` as the user's email string. Your database would be corrupted, and you wouldn't know until users complained. By throwing a `ValidationError`, Pydantic forces your application to fail instantly and loudly, alerting you to the broken integration.

---

### Interview Prep
<details>
<summary><b> Question 1: Explain the difference between Lax Mode and Strict Mode in Pydantic. When would you use each? <b> (Click to reveal)</summary>
<br>
    
**Answer:** By default, Pydantic runs in Lax Mode, meaning it will attempt logical type coercions—like converting the string `"1.5"` to a float, or `"false"` to a boolean. This is incredibly useful for parsing JSON or URL query parameters where everything arrives as a string.

Strict Mode disables this coercion. If you require an `int` and the client sends a `float` or `string`, Pydantic immediately throws a `ValidationError`. I use Strict Mode in high-stakes environments—like financial transaction APIs or healthcare records—where implicit type coercion could lead to rounding errors or data loss.

</details>

---
<details>
<summary><b> Question 2: If a Pydantic model requires a `str` field, and I pass the integer `123` in Lax Mode, what happens and why? <b> (Click to reveal)</summary>
<br>
    
 **Answer:** It throws a `ValidationError`. Even in Lax mode, Pydantic refuses to coerce standard types like integers, floats, or dictionaries into strings.

The reasoning is bug prevention. In Python, every object has a string representation. If Pydantic auto-coerced to strings, a silent API change (e.g., a field changing from a string to a nested dictionary) would be silently converted into a stringified dictionary rather than failing. Pydantic forces these mismatches to fail loudly so you can catch integration bugs immediately.   

</details>

---
<details>
<summary><b> Question 3: If you need to accept either a string OR an integer for a specific field, but you ultimately want it stored as a string, how do you handle that if Pydantic rejects int-to-string coercion? <b> (Click to reveal)</summary>
<br>
    
**Answer:** Instead of relying on implicit coercion, I would explicitly define the field to accept both using the `Union` type (or the `|` operator in newer Python versions), and then use a custom validator to cast it.

I'd define the field as `field: str | int`. Then, I'd write a `@field_validator('field')` that takes the value and explicitly calls `str(value)`. This makes the coercion intentional and documented in the code, rather than relying on the framework's implicit background rules.   

</details>





In [8]:
from pydantic import BaseModel

class Coordinates(BaseModel):
    x: float
    y: float

# Deserializing with mismatched types
# 'x' is an int, 'y' is a string
p2 = Coordinates.model_validate({"x": 0, "y": "1.2"})

print(type(p2.x)) # <class 'float'> -> Coerced from 0
print(type(p2.y)) # <class 'float'> -> Coerced from "1.2"

<class 'float'>
<class 'float'>
